## 3. Generating Synthetic Multi-View Anomaly Data

You must complete the previous setup step [0-setup-cuda128.ipynb](./0-setup-cuda128.ipynb) and training step [1-training-multiview.ipynb](./1-training-multiview.ipynb).

**Important**: Multi-view inference is very similar to single-view inference. The main difference is the JSONL format, which now requires lists of image/mask filenames for each view.


### 3.0 Setting Up the Environment

This notebook requires the user to set the environment variable `LOCAL_PROJECT_DIR` to the path of the PAIDF AnomalyGen repo. Remember to replace `FIXME` placeholder below with the correct path.


In [ ]:
# Set `LOCAL_PROJECT_DIR` for PAIDF AnomalyGen.
LOCAL_PROJECT_DIR="FIXME"
# Set the working directory to this path for the shell.
%cd {LOCAL_PROJECT_DIR}
# Use `cd ${LOCAL_PROJECT_DIR}` if you are copy-pasting this into a terminal.


### 3.1 Multi-View Generation Configuration

Multi-view generation uses a `.jsonl` file similar to single-view, but with **lists** for `image_filenames` and `mask_filename`.

#### Key Differences from Single-View:

1. **`image_filenames`**: List of image paths (one per view), e.g., `["path/view0.png", "path/view1.png", ...]`
2. **`mask_filename`**: 
   - **Shared mask**: Single string path, e.g., `"path/mask.png"`
   - **Per-view mask**: List of mask paths (one per view), e.g., `["path/view0_mask.png", "path/view1_mask.png", ...]`

#### Example 1: Shared Mask (PeppermintCandy)

```json
{
  "mask_filename": "datasets/PeppermintCandy_multiview/PeppermintCandy/mask/bumps/25_bumps_mask.png",
  "anomaly_type": "PeppermintCandy+bumps",
  "image_filenames": [
    "datasets/PeppermintCandy_multiview/PeppermintCandy/anomaly_image/bumps/25_bumps_view0.png",
    "datasets/PeppermintCandy_multiview/PeppermintCandy/anomaly_image/bumps/25_bumps_view1.png",
    "datasets/PeppermintCandy_multiview/PeppermintCandy/anomaly_image/bumps/25_bumps_view2.png",
    "datasets/PeppermintCandy_multiview/PeppermintCandy/anomaly_image/bumps/25_bumps_view3.png",
    "datasets/PeppermintCandy_multiview/PeppermintCandy/anomaly_image/bumps/25_bumps_view4.png",
    "datasets/PeppermintCandy_multiview/PeppermintCandy/anomaly_image/bumps/25_bumps_view5.png"
  ],
  "guidance": 1.5,
  "num_steps": 35,
  "crop_and_paste": false,
  "iteration_generation_max_instance": 1
}
```

#### Example 2: Per-View Mask (SimCardSet)

```json
{
  "mask_filename": [
    "datasets/SimCardSet/SimCardSet/mask/CH/S0001_view1_mask.png",
    "datasets/SimCardSet/SimCardSet/mask/CH/S0001_view2_mask.png",
    "datasets/SimCardSet/SimCardSet/mask/CH/S0001_view3_mask.png",
    "datasets/SimCardSet/SimCardSet/mask/CH/S0001_view4_mask.png",
    "datasets/SimCardSet/SimCardSet/mask/CH/S0001_view5_mask.png"
  ],
  "anomaly_type": "SimCardSet+CH",
  "image_filenames": [
    "datasets/SimCardSet/SimCardSet/anomaly_image/CH/S0001_view1.jpg",
    "datasets/SimCardSet/SimCardSet/anomaly_image/CH/S0001_view2.jpg",
    "datasets/SimCardSet/SimCardSet/anomaly_image/CH/S0001_view3.jpg",
    "datasets/SimCardSet/SimCardSet/anomaly_image/CH/S0001_view4.jpg",
    "datasets/SimCardSet/SimCardSet/anomaly_image/CH/S0001_view5.jpg"
  ],
  "guidance": 1.5,
  "num_steps": 35,
  "crop_and_paste": false,
  "iteration_generation_max_instance": 1
}
```

**Note**: All other fields (guidance, num_steps, crop_and_paste, etc.) work the same as single-view.


### 3.2 Multi-View Generation Commands

The generation command uses `scripts.anomaly_gen.multiview_synthetic_dataset_generation` instead of `synthetic_dataset_generation`.

#### Example 1: PeppermintCandy (Shared Mask)


<details>
<summary> <b> The equivalent command in the bash terminal. (Click to show) <b> </summary>

```bash
export IMAGINAIRE_OUTPUT_ROOT=./results && \
CUDA_HOME=$CONDA_PREFIX \
CUDA_VISIBLE_DEVICES=0 \
torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.multiview_synthetic_dataset_generation \
    --config=cosmos_predict2/configs/base/ag_config.py \
    --ag_checkpoint_dir=results/anomaly_gen/PeppermintCandy/PeppermintCandy_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe_augmeted0 \
    --step=200 \
    --input_data_path=ag_inference/peppermint_inference.jsonl \
    --output_image_path=results/PeppermintCandy/multiview_output \
    --seed=0 \
    -- experiment=predict2_anomaly_gen_multiview_ddp_2b
```

</details>


In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "export IMAGINAIRE_OUTPUT_ROOT=./results && \
        CUDA_HOME=\$CONDA_PREFIX \
        CUDA_VISIBLE_DEVICES=0 \
        torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.multiview_synthetic_dataset_generation \
        --config=cosmos_predict2/configs/base/ag_config.py \
        --ag_checkpoint_dir=results/anomaly_gen/PeppermintCandy/PeppermintCandy_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe_augmeted0 \
        --step=150000 \
        --input_data_path=ag_inference/peppermint_inference.jsonl \
        --output_image_path=results/PeppermintCandy/multiview_output \
        --seed=0 \
        -- experiment=predict2_anomaly_gen_multiview_ddp_2b"


#### Example 2: SimCardSet (Per-View Mask)


<details>
<summary> <b> The equivalent command in the bash terminal. (Click to show) <b> </summary>

```bash
export IMAGINAIRE_OUTPUT_ROOT=./results && \
CUDA_HOME=$CONDA_PREFIX \
CUDA_VISIBLE_DEVICES=0 \
torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.multiview_synthetic_dataset_generation \
    --config=cosmos_predict2/configs/base/ag_config.py \
    --ag_checkpoint_dir=results/anomaly_gen/SimCardSet/SimCardSet_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe \
    --step=200 \
    --input_data_path=ag_inference/simcardset_validation.jsonl \
    --output_image_path=results/SimCardSet/multiview_output \
    --seed=0 \
    -- experiment=predict2_anomaly_gen_multiview_ddp_2b
```

</details>


In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "export IMAGINAIRE_OUTPUT_ROOT=./results && \
        CUDA_HOME=\$CONDA_PREFIX \
        CUDA_VISIBLE_DEVICES=0 \
        torchrun --nproc_per_node=1 --master_port=12351 -m scripts.anomaly_gen.multiview_synthetic_dataset_generation \
        --config=cosmos_predict2/configs/base/ag_config.py \
        --ag_checkpoint_dir=results/anomaly_gen/SimCardSet/SimCardSet_multiview_2B_512_perviewEmbed_token208_length317_dataTypeImage_inheritModel_inheritPipe \
        --step=150000 \
        --input_data_path=ag_inference/simcardset_validation.jsonl \
        --output_image_path=results/SimCardSet/multiview_output \
        --seed=0 \
        -- experiment=predict2_anomaly_gen_multiview_ddp_2b"


### 3.3 Generation Output Structure

Multi-view generation outputs are stored per-view:

```
<output_image_path>/
├── original_image/
│   ├── <anomaly>_00000_view0.png
│   ├── <anomaly>_00000_view1.png
│   └── ...
├── original_mask/
│   ├── <anomaly>_00000_view0.png
│   ├── <anomaly>_00000_view1.png
│   └── ...
├── reconstructed_image/
│   ├── <anomaly>_00000_view0.png
│   ├── <anomaly>_00000_view1.png
│   └── ...
├── cropped_image/  # If crop_and_paste=True
│   ├── <anomaly>_00000_view0_0.png  # [view]_[instance]
│   └── ...
├── cropped_mask/
│   ├── <anomaly>_00000_view0_0.png
│   └── ...
├── annotated_image/
│   ├── <anomaly>_00000_view0_0.png
│   └── ...
└── SDG_result.csv  # Generation metadata
```

**Note**: 
- Each view's results are saved separately
- `cropped_image`, `cropped_mask`, and `annotated_image` may have instance indices (e.g., `_0`, `_1`) if multiple instances are generated
- `cropped_mask` and `annotated_image` use the **intersection mask** (actual denoise condition) rather than the union mask, to clearly show what each view actually sees


### 3.4 Multi-View Specific Features

#### Union-Based Mask Splitting

Multi-view uses a **union-based mask splitting** strategy:

1. **Compute union** of all view masks → "super mask"
2. **Split** the union mask according to `iteration_generation_max_instance`
3. **Intersect** each split mask with each view's original mask → view-specific denoise condition

This ensures:
- **Consistent augmentation** across views (crop region determined by union mask)
- **View-specific denoise condition** (each view uses its actual visible mask region)

## Summary

Multi-view training and inference are very similar to single-view, with these key differences:

1. **Dataset structure**: Multiple view images per sample, with shared or per-view masks
2. **Config**: Add `view_types` list to dataset configuration
3. **JSONL format**: Use lists for `image_filenames` and `mask_filename` (for per-view masks)
4. **Script**: Use `multiview_synthetic_dataset_generation` instead of `synthetic_dataset_generation`
5. **Output**: Results are saved per-view with view indices in filenames

Everything else (guidance, num_steps, crop_and_paste, augmentation, etc.) works exactly the same as single-view!
